In [5]:
import bilby
import numpy as np

In [6]:
def bbh_lensing(frequency_array, mass_1, mass_2, a_1, a_2, tilt_1, tilt_2, phi_12, phi_jl, 
                        luminosity_distance, theta_jn, phase, 
                        delta_theta_jn, delta_phase, delta_t, rel_mu, delta_z,
                        **kwargs):
    waveform_kwargs = dict(
        waveform_approximant='IMRPhenomPv2', reference_frequency=50.0,
        minimum_frequency=20.0, maximum_frequency=frequency_array[-1],
        catch_waveform_errors=False, pn_spin_order=-1, pn_tidal_order=-1,
        pn_phase_order=-1, pn_amplitude_order=0)
    waveform_kwargs.update(kwargs)

    # plus waveform
    wf_plus = bilby.gw.source._base_lal_cbc_fd_waveform(
        frequency_array=frequency_array, mass_1=mass_1, mass_2=mass_2,
        luminosity_distance=luminosity_distance, theta_jn=theta_jn, phase=phase,
        a_1=a_1, a_2=a_2, tilt_1=tilt_1, tilt_2=tilt_2, phi_12=phi_12,
        phi_jl=phi_jl, **waveform_kwargs)
    hplus_plus = wf_plus['plus']
    hcross_plus = wf_plus['cross']

    # minus waveform, with different parameters
    wf_minus = bilby.gw.source._base_lal_cbc_fd_waveform(
        frequency_array=frequency_array, mass_1=mass_1, mass_2=mass_2,
        luminosity_distance=luminosity_distance, theta_jn=theta_jn+delta_theta_jn, phase=phase+delta_phase,
        a_1=a_1, a_2=a_2, tilt_1=tilt_1, tilt_2=tilt_2, phi_12=phi_12,
        phi_jl=phi_jl, **waveform_kwargs)
    hplus_minus = wf_minus['plus']
    hcross_minus = wf_minus['cross']

    # phase shift due to time delay between the two images
    F_arr = np.asarray([np.sqrt(np.abs(rel_mu)) * np.exp(complex(0, -f * delta_t)) for f in frequency_array])
    # frequency shift due to Doppler effect
    
    # full waveform
    hplus = hplus_plus + rel_mu * F_arr * hplus_minus
    hcross = hcross_plus + rel_mu * F_arr * hcross_minus

    return {"plus": hplus, "cross": hcross}

In [7]:
injection_parameters = dict(
    mass_1=36.0,
    mass_2=29.0,
    a_1=0.4,
    a_2=0.3,
    tilt_1=0.5,
    tilt_2=1.0,
    phi_12=1.7,
    phi_jl=0.3,
    luminosity_distance=4000.0,
    theta_jn=0.4,
    psi=2.659,
    phase=1.3,
    geocent_time=1126259642.413,
    ra=1.375,
    dec=-1.2108,
    delta_theta_jn=0.1,
    delta_phase=0.1,
    delta_t=0.5,
    rel_mu=3
)

duration = 2
sampling_frequency = 2048
outdir = "outdir"
label = "bbh_lensing_model"

waveform_arguments = dict(
    waveform_approximant="IMRPhenomPv2",
    reference_frequency=20,
    minimum_frequency=20,
    catch_waveform_errors=True,
)
wf_generator = bilby.gw.waveform_generator.WaveformGenerator(
    duration=duration,
    sampling_frequency=sampling_frequency,
    start_time=0,
    frequency_domain_source_model=bbh_lensing,
    time_domain_source_model=None,
    parameters=None,
    parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters,
    waveform_arguments=waveform_arguments,
)

20:26 bilby INFO    : Waveform generator initiated with
  frequency_domain_source_model: __main__.bbh_lensing
  time_domain_source_model: None
  parameter_conversion: bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters


In [8]:
ifos = bilby.gw.detector.InterferometerList(["H1", "L1"])
ifos.set_strain_data_from_power_spectral_densities(
    sampling_frequency=sampling_frequency,
    duration=duration,
    start_time=injection_parameters["geocent_time"] - 2,
)
ifos.inject_signal(
    waveform_generator=wf_generator, parameters=injection_parameters
)

20:26 bilby INFO    : Injected signal in H1:
20:26 bilby INFO    :   optimal SNR = 29.97
20:26 bilby INFO    :   matched filter SNR = 29.51+0.62j
20:26 bilby INFO    :   mass_1 = 36.0
20:26 bilby INFO    :   mass_2 = 29.0
20:26 bilby INFO    :   a_1 = 0.4
20:26 bilby INFO    :   a_2 = 0.3
20:26 bilby INFO    :   tilt_1 = 0.5
20:26 bilby INFO    :   tilt_2 = 1.0
20:26 bilby INFO    :   phi_12 = 1.7
20:26 bilby INFO    :   phi_jl = 0.3
20:26 bilby INFO    :   luminosity_distance = 4000.0
20:26 bilby INFO    :   theta_jn = 0.4
20:26 bilby INFO    :   psi = 2.659
20:26 bilby INFO    :   phase = 1.3
20:26 bilby INFO    :   geocent_time = 1126259642.413
20:26 bilby INFO    :   ra = 1.375
20:26 bilby INFO    :   dec = -1.2108
20:26 bilby INFO    :   delta_theta_jn = 0.1
20:26 bilby INFO    :   delta_phase = 0.1
20:26 bilby INFO    :   delta_t = 0.5
20:26 bilby INFO    :   rel_mu = 3
20:26 bilby INFO    : Injected signal in L1:
20:26 bilby INFO    :   optimal SNR = 24.23
20:26 bilby INFO    : 

[{'plus': array([0.+0.j, 0.-0.j, 0.-0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j]),
  'cross': array([0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j])},
 {'plus': array([0.+0.j, 0.-0.j, 0.-0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j]),
  'cross': array([0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j])}]

In [9]:
priors = bilby.gw.prior.BBHPriorDict()
for key in [
    "a_1",
    "a_2",
    "tilt_1",
    "tilt_2",
    "phi_12",
    "phi_jl",
    "phase",
    "theta_jn",
    "psi",
    "luminosity_distance",
    "ra",
    "dec",
    "geocent_time",
    "delta_t",
    "rel_mu",
]:
    priors[key] = injection_parameters[key]

priors["delta_theta_jn"] = bilby.core.prior.Uniform(
    name="delta_theta_jn", minimum=0, maximum=1, unit="rad"
)
priors["delta_phase"] = bilby.core.prior.Uniform(
    name="delta_phase", minimum=0, maximum=1, unit="rad"
)

20:26 bilby INFO    : No prior given, using default BBH priors in /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/bilby/gw/prior_files/precessing_spins_bbh.prior.


In [10]:
likelihood = bilby.gw.GravitationalWaveTransient(
    interferometers=ifos, waveform_generator=wf_generator
)

In [11]:
result = bilby.run_sampler(
    likelihood=likelihood,
    priors=priors,
    sampler="dynesty",
    outdir=outdir,
    injection_parameters=injection_parameters,
    label="bbh_lensing",
)
result.plot_corner()

20:26 bilby INFO    : Running for label 'bbh_lensing', output will be saved to 'outdir'
20:26 bilby INFO    : Using lal version 7.5.0
20:26 bilby INFO    : Using lal git version Branch: None;Tag: lalsuite-v7.22;Id: 5c23000593918e5e4ef2ff809eccc1722b5d0795;;Builder: Duncan Macleod <duncan.macleod@ligo.org>;Repository status: CLEAN: All modifications committed
20:26 bilby INFO    : Using lalsimulation version 5.4.0
20:26 bilby INFO    : Using lalsimulation git version Branch: None;Tag: lalsuite-v7.22;Id: 5c23000593918e5e4ef2ff809eccc1722b5d0795;;Builder: Duncan Macleod <duncan.macleod@ligo.org>;Repository status: CLEAN: All modifications committed
20:26 bilby INFO    : Analysis priors:
20:26 bilby INFO    : mass_ratio=bilby.gw.prior.UniformInComponentsMassRatio(minimum=0.125, maximum=1, name='mass_ratio', latex_label='$q$', unit=None, boundary=None, equal_mass=False)
20:26 bilby INFO    : chirp_mass=bilby.gw.prior.UniformInComponentsChirpMass(minimum=25, maximum=100, name='chirp_mass', l

217it [00:24,  9.69it/s, bound:0 nc:  2 ncall:1.6e+03 eff:13.7% logz-ratio=-1154.31+/-0.12 dlogz:1817.895>0.1]

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/dynesty/sampler.py:761: UserWarning: The sampling was stopped short due to maxiter/maxcall limit the delta(log(z)) criterion is not achieved; posterior may be poorly sampled
  warnings.warn('The sampling was stopped short due to'


955it [10:32,  1.45s/it, bound:19 nc:  1 ncall:1.9e+04 eff:7.3% logz-ratio=23.78+/-0.15 dlogz:728.947>0.1]    

20:37 bilby INFO    : Written checkpoint file outdir/bbh_lensing_resume.pickle
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/dynesty/plotting.py:786: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set_ylim([0., max(y0) * 1.05])
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/dynesty/plotting.py:786: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set_ylim([0., max(y0) * 1.05])
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/dynesty/plotting.py:252: RuntimeWarning: overflow encountered in exp
  np.exp(logwt), logz if logplot else np.exp(logz)
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/dynesty/plotting.py:282: RuntimeWarning: overflow encountered in exp
  zspan = (0., 1.05 * np.exp(logz[-1] + 3. * logzerr[-1]))

...

1665it [41:51,  3.33s/it, bound:88 nc:  1 ncall:6.3e+04 eff:3.3% logz-ratio=346.42+/-0.11 dlogz:404.802>0.1] 

21:08 bilby INFO    : Written checkpoint file outdir/bbh_lensing_resume.pickle
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/dynesty/plotting.py:786: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set_ylim([0., max(y0) * 1.05])
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/dynesty/plotting.py:786: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set_ylim([0., max(y0) * 1.05])
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/dynesty/plotting.py:252: RuntimeWarning: overflow encountered in exp
  np.exp(logwt), logz if logplot else np.exp(logz)
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/dynesty/plotting.py:282: RuntimeWarning: overflow encountered in exp
  zspan = (0., 1.05 * np.exp(logz[-1] + 3. * logzerr[-1]))

1686it [42:38,  2.54s/it, bound:90 nc:  1 ncall:6.4e+04 eff:3.1% logz-ratio=164.08+/-0.15 dlogz:588.032>0.1]

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/dynesty/sampler.py:761: UserWarning: The sampling was stopped short due to maxiter/maxcall limit the delta(log(z)) criterion is not achieved; posterior may be poorly sampled
  warnings.warn('The sampling was stopped short due to'


1833it [48:16,  1.85s/it, bound:101 nc:  1 ncall:7.1e+04 eff:2.6% logz-ratio=48.19+/-0.13 dlogz:704.401>0.1]